In [7]:
import pandas as pd
import folium
import json
from shapely.geometry import shape, Polygon
from shapely.ops import unary_union
import ipywidgets as widgets
from IPython.display import display

# === 1. Load data agregasi ===
df_agg = pd.read_csv("../server/data/podes_aggregated_per_desa.csv")

# === 2. Load geojson ===
with open("../server/data/kelurahan.geojson", "r", encoding="utf-8") as f:
    desa_geo = json.load(f)

# Samakan nama desa
df_agg["NAMA_DESA"] = df_agg["NAMA_DESA"].str.strip().str.upper()
for feature in desa_geo["features"]:
    feature["properties"]["nm_kelurahan"] = feature["properties"]["nm_kelurahan"].strip().upper()

# === 3. Buat masking untuk luar Kota Batu ===
polygons = [shape(feature["geometry"]) for feature in desa_geo["features"]]
kota_batu_polygon = unary_union(polygons)
bounds = Polygon([(110, -9), (115, -9), (115, -5), (110, -5)])
mask = bounds.difference(kota_batu_polygon)

# === 4. Daftar indikator (nama ditampilkan = kode aslinya) ===
indikator_list = ["R502A", "R502B", "R503A", "R503C", "R504A2", "R504A3", "R504A4", "R504C", "R504D", "R504E", "R504F1", "R505", "R511C1", "R511C2A", "R511C2B", "R511C2C", "R511C3", "R514AK2", "R514AK3", "R514AK4", "R514CK2", "R514CK3", "R514CK4", "R515B", "R516", "R517", "R601AK2", "R601AK3", "R601AK4", "R601AK5", "R601AK6", "R601BK2", "R601BK3", "R601BK4", "R601BK5", "R601BK6", "R601DK2", "R601DK3", "R601DK4", "R601DK5", "R601DK6", "R601GK2", "R601GK3", "R601GK4", "R601GK5", "R601GK6", "R601IK2", "R601IK3", "R601IK4", "R601IK5", "R601IK6", "R604A", "R604C", "R604D", "R6061", "R6062", "R6063", "R701BK2", "R701BK3", "R701BK4", "R701BK5", "R701DK2", "R701DK3", "R701DK4", "R701DK5", "R701FK2", "R701FK3", "R701FK4", "R701FK5", "R701HK2", "R701HK3", "R701HK4", "R701HK5", "R704AK2", "R704CK2", "R704DK2", "R1001C", "R1005A", "R1005C", "R1005D"]

# Buat dictionary otomatis: label = kode, value = kode
indikator_dict = {kode: kode for kode in indikator_list}

# === 5. Dropdown widget ===
dropdown = widgets.Dropdown(
    options=list(indikator_dict.keys()),
    value="R701DK2",  # default contoh: Jumlah SD
    description="Pilih Indikator:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# === 6. Fungsi update peta ===
def update_map(indikator_name):
    indikator_col = indikator_dict[indikator_name]

    # Buat peta dasar
    map_batu = folium.Map(location=[-7.8671, 112.5239], zoom_start=12, tiles="CartoDB positron")

    choropleth = folium.Choropleth(
        geo_data=desa_geo,
        data=df_agg,
        columns=["NAMA_DESA", indikator_col],
        key_on="feature.properties.nm_kelurahan",
        fill_color="YlOrRd",
        fill_opacity=0.7,
        line_opacity=0.5,
        legend_name=indikator_col
    ).add_to(map_batu)

    # Tooltip nama desa
    folium.GeoJson(
        desa_geo,
        style_function=lambda x: {"fillColor": "transparent", "color": "black", "weight": 1},
        tooltip=folium.GeoJsonTooltip(fields=["nm_kelurahan"], aliases=["Desa: "])
    ).add_to(map_batu)

    # Masking polygon
    folium.GeoJson(
        mask.__geo_interface__,
        style_function=lambda x: {
            "fillColor": "lightgray",
            "color": "lightgray",
            "weight": 1,
            "fillOpacity": 0.6
        }
    ).add_to(map_batu)

    # Zoom otomatis
    map_batu.fit_bounds(choropleth.geojson.get_bounds())

    display(map_batu)

# === 7. Hubungkan dropdown dengan fungsi update ===
widgets.interact(update_map, indikator_name=dropdown)

interactive(children=(Dropdown(description='Pilih Indikator:', index=61, layout=Layout(width='400px'), options…

<function __main__.update_map(indikator_name)>